# 📊 Phân Tích Tập Dữ Liệu Huấn Luyện NER (`train.jsonl`)

Notebook này thực hiện phân tích chi tiết tập dữ liệu gán nhãn `train.jsonl` dùng để huấn luyện mô hình nhận diện thực thể tên (Named Entity Recognition - NER) cho thông tin CV ứng viên. Quá trình phân tích bao gồm:
1. **Tổng quan dữ liệu**: Thống kê số lượng CV, tokens, độ dài.
2. **Đánh giá chất lượng**: Kiểm tra sự trùng khớp chiều dài giữa tokens và tags, kiểm tra tính nhất quán của quy tắc BIO.
3. **Phân tích nhãn NER**: Xem xét phân bố của các thực thể gán nhãn, phát hiện mất cân bằng dữ liệu.
4. **Trích xuất nội dung**: Phân tích các thực thể thực tế (SKILL, COMPANY, UNIVERSITY,...) xuất hiện nhiều nhất.
5. **Phân tích ngôn ngữ**: Ước lượng tỷ lệ ngôn ngữ Tiếng Anh và Tiếng Việt trong bộ dữ liệu.

In [ ]:
import json
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Cấu hình trực quan hóa
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["axes.unicode_minus"] = False

# Palette màu đẹp mắt, hiện đại
PALETTE = sns.color_palette("viridis", 10)
sns.set_palette("muted")

## 1. Đọc và Kiểm tra Dữ liệu Cơ bản

In [ ]:
# Xác định đường dẫn file dữ liệu
data_paths = [
    "../data/annotated/train.jsonl",
    "data/annotated/train.jsonl",
    "recruitment-ai/data/annotated/train.jsonl"
]

file_path = None
for p in data_paths:
    if os.path.exists(p):
        file_path = p
        break

if not file_path:
    raise FileNotFoundError("Không tìm thấy file train.jsonl! Hãy kiểm tra lại thư mục chạy notebook.")

print(f"Đang đọc dữ liệu từ: {file_path}")
data = []
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))

print(f"✓ Đã nạp thành công {len(data)} mẫu CV.")

## 2. Thống kê Mô tả Tổng quan

In [ ]:
num_samples = len(data)
token_lens = [len(x["tokens"]) for x in data]
total_tokens = sum(token_lens)
vocab = set()
for x in data:
    vocab.update([t.lower() for t in x["tokens"]])

print("=== THỐNG KÊ TỔNG QUAN ===")
print(f"- Tổng số mẫu CV: {num_samples}")
print(f"- Tổng số token: {total_tokens:,}")
print(f"- Kích thước từ vựng (vocabulary size): {len(vocab):,}")
print(f"- Số lượng token trung bình mỗi CV: {np.mean(token_lens):.1f}")
print(f"- Trung vị độ dài CV: {np.median(token_lens):.1f} tokens")
print(f"- CV ngắn nhất: {np.min(token_lens)} tokens")
print(f"- CV dài nhất: {np.max(token_lens)} tokens")
print(f"- Độ lệch chuẩn độ dài (std dev): {np.std(token_lens):.1f}")

## 3. Kiểm tra Chất lượng Dữ liệu

### 3.1. Kiểm tra Lệch Chiều dài Tokens và Tags
Lỗi này xảy ra khi số lượng tokens khác số lượng nhãn NER, gây lỗi khi đưa dữ liệu vào huấn luyện các mô hình như PhoBERT.

In [ ]:
mismatches = []
for idx, item in enumerate(data):
    len_t = len(item["tokens"])
    len_n = len(item["ner_tags"])
    if len_t != len_n:
        mismatches.append({
            "index": idx + 1,
            "id": item.get("id", f"sample_{idx}"),
            "tokens_len": len_t,
            "tags_len": len_n,
            "diff": len_t - len_n
        })

print(f"=== KẾT QUẢ KIỂM TRA LỆCH ĐỘ DÀI ===")
if mismatches:
    print(f"❌ Phát hiện {len(mismatches)} mẫu có độ dài tokens và tags không khớp nhau!")
    mismatches_df = pd.DataFrame(mismatches)
    display(mismatches_df)
else:
    print("✅ Tuyệt vời! Không phát hiện mẫu nào bị lệch độ dài tokens và tags.")

### 3.2. Kiểm tra Tính Nhất Quán của Quy Tắc BIO
Quy tắc BIO yêu cầu nhãn bắt đầu bằng `B-` trước khi tiếp tục bằng `I-` cùng loại. Việc có `I-` xuất hiện sau `O` hoặc sau một thực thể loại khác là không nhất quán.

In [ ]:
bio_errors = []
for idx, item in enumerate(data):
    tags = item["ner_tags"]
    prev_tag = "O"
    for pos, tag in enumerate(tags):
        if tag.startswith("I-"):
            ent_type = tag.split("-")[1]
            if prev_tag == "O":
                bio_errors.append({
                    "id": item.get("id", f"sample_{idx}"),
                    "position": pos,
                    "token": item["tokens"][pos],
                    "error_tag": tag,
                    "prev_tag": prev_tag,
                    "type": "I-tag sau O (Thiếu B-tag)"
                })
            elif prev_tag.startswith("B-") or prev_tag.startswith("I-"):
                prev_type = prev_tag.split("-")[1]
                if prev_type != ent_type:
                    bio_errors.append({
                        "id": item.get("id", f"sample_{idx}"),
                        "position": pos,
                        "token": item["tokens"][pos],
                        "error_tag": tag,
                        "prev_tag": prev_tag,
                        "type": f"I-tag sai loại thực thể (sau {prev_tag})"
                    })
        prev_tag = tag

print("=== KẾT QUẢ KIỂM TRA QUY TẮC BIO ===")
if bio_errors:
    print(f"⚠️ Tìm thấy {len(bio_errors)} vị trí vi phạm quy tắc BIO tagging.")
    bio_df = pd.DataFrame(bio_errors)
    print(f"Một số lỗi tiêu biểu (tối đa 10 dòng):")
    display(bio_df.head(10))
else:
    print("✅ Tuyệt vời! Bộ dữ liệu hoàn toàn tuân thủ quy tắc BIO tagging.")

## 4. Phân Tích Trực Quan Hóa Dữ Liệu

### 4.1. Phân bố Độ dài CV

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(token_lens, bins=40, kde=True, color="#34495e", edgecolor="black", alpha=0.7)
plt.axvline(np.mean(token_lens), color="red", linestyle="--", linewidth=2, label=f"Mean: {np.mean(token_lens):.1f}")
plt.axvline(np.median(token_lens), color="orange", linestyle="-", linewidth=2, label=f"Median: {np.median(token_lens):.1f}")
plt.title("Phân Phối Độ Dài CV (Số lượng Tokens)", fontsize=15, fontweight="bold", pad=15)
plt.xlabel("Số lượng Tokens", fontsize=12)
plt.ylabel("Số lượng CV (Mẫu)", fontsize=12)
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

### 4.2. Phân bố Nhãn NER ở Cấp độ Token
Thống kê tổng số lượng token tương ứng với từng nhãn để đánh giá sự phân bố.

In [ ]:
all_tags = []
for x in data:
    all_tags.extend(x["ner_tags"])

tag_counts = Counter(all_tags)
total_tokens_count = len(all_tags)

tag_stats = []
for tag, count in tag_counts.items():
    tag_stats.append({
        "Tag": tag,
        "Count": count,
        "Percentage (%)": (count / total_tokens_count) * 100
    })

tag_stats_df = pd.DataFrame(tag_stats).sort_values(by="Count", ascending=False)
print(f"Tổng số nhãn token: {total_tokens_count}")
display(tag_stats_df)

### 4.3. Biểu đồ Phân Bố Thực Thể (Không tính nhãn 'O')
Nhãn `O` (nhãn ngoài thực thể) thường chiếm đa số lớn. Để nhìn rõ phân bố của các thực thể đích, chúng ta loại bỏ `O`.

In [ ]:
# Lọc bỏ nhãn 'O'
entities_tags_only = [t for t in all_tags if t != "O"]
entity_counts = Counter(entities_tags_only)

entity_df = pd.DataFrame(entity_counts.items(), columns=["Entity Tag", "Count"]).sort_values(by="Count", ascending=False)

plt.figure(figsize=(14, 7))
sns.barplot(data=entity_df, x="Count", y="Entity Tag", hue="Entity Tag", palette="viridis", legend=False)
plt.title("Tần Suất Xuất Hiện Của Các Nhãn Thực Thể (Loại bỏ 'O')", fontsize=15, fontweight="bold", pad=15)
plt.xlabel("Số lượng Token được gán nhãn", fontsize=12)
plt.ylabel("Nhãn Thực Thể", fontsize=12)
plt.tight_layout()
plt.show()

### 4.4. Phân Tích Tỷ Lệ Ngôn Ngữ
Chúng ta phân biệt CV tiếng Việt và tiếng Anh dựa trên sự xuất hiện của các chữ cái có dấu tiếng Việt trong tokens.

In [ ]:
vietnamese_chars = re.compile(r"[àáảãạâầấẩẫậăằắẳẵặèéẻẽẹêềếểễệđìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵ]", re.IGNORECASE)

vietnamese_cv_count = 0
english_cv_count = 0

for item in data:
    text = " ".join(item["tokens"])
    if vietnamese_chars.search(text):
        vietnamese_cv_count += 1
    else:
        english_cv_count += 1

lang_df = pd.DataFrame({
    "Ngôn ngữ": ["Tiếng Anh (English)", "Tiếng Việt (Vietnamese)"],
    "Số lượng CV": [english_cv_count, vietnamese_cv_count]
})

print(f"- Số CV Tiếng Anh: {english_cv_count} ({english_cv_count/num_samples*100:.1f}%)")
print(f"- Số CV Tiếng Việt: {vietnamese_cv_count} ({vietnamese_cv_count/num_samples*100:.1f}%)")

# Trực quan hóa bằng biểu đồ tròn
plt.figure(figsize=(8, 8))
plt.pie(lang_df["Số lượng CV"], labels=lang_df["Ngôn ngữ"], autopct="%1.1f%%", 
        startangle=140, colors=["#3498db", "#e74c3c"], textprops={"fontsize": 14, "weight": "bold"})
plt.title("Tỷ Lệ Ngôn Ngữ Các CV Trong Tập Huấn Luyện", fontsize=16, fontweight="bold", pad=20)
plt.show()

## 5. Trích Xuất và Phân Tích Thực Thể Cụ Thể (Full Entity Extraction)

Chúng ta sẽ xây dựng hàm trích xuất để gộp các token liên tiếp cùng nhóm thực thể (như `B-SKILL` và `I-SKILL`) thành một thực thể trọn vẹn, nhằm hiểu nội dung cụ thể được gán nhãn.

In [ ]:
def extract_full_entities(record):
    tokens = record["tokens"]
    tags = record["ner_tags"]
    entities = []
    
    current_entity = []
    current_type = None
    
    for token, tag in zip(tokens, tags):
        if tag == "O":
            if current_entity:
                entities.append((" ".join(current_entity), current_type))
                current_entity = []
                current_type = None
        elif tag.startswith("B-"):
            if current_entity:
                entities.append((" ".join(current_entity), current_type))
            current_type = tag.split("-")[1]
            current_entity = [token]
        elif tag.startswith("I-"):
            entity_type = tag.split("-")[1]
            if current_type == entity_type:
                current_entity.append(token)
            else:
                if current_entity:
                    entities.append((" ".join(current_entity), current_type))
                current_type = entity_type
                current_entity = [token]
                
    if current_entity:
        entities.append((" ".join(current_entity), current_type))
        
    return entities

# Trích xuất toàn bộ thực thể
all_entities = []
for r in data:
    all_entities.extend(extract_full_entities(r))

print(f"Tổng số thực thể đầy đủ trích xuất được từ {len(data)} CV: {len(all_entities):,}")

### 5.1. Phân Phối Thực Thể Gộp Nhóm theo Loại (Entity Count by Type)

In [ ]:
entity_types = [ent_type for _, ent_type in all_entities]
entity_type_counts = Counter(entity_types)

ent_type_df = pd.DataFrame(entity_type_counts.items(), columns=["Loại thực thể", "Số lượng"]).sort_values(by="Số lượng", ascending=False)
display(ent_type_df)

### 5.2. Các Giá Trị Phổ Biến Nhất Của Từng Thực Thể
Chúng ta sẽ xem top các giá trị xuất hiện nhiều nhất cho các thực thể quan trọng: `SKILL`, `UNIVERSITY`, `COMPANY`, `JOB_TITLE`.

In [ ]:
# Gộp nhóm thực thể theo loại và chuẩn hóa văn bản
entity_by_type = {}
for val, ent_type in all_entities:
    if ent_type not in entity_by_type:
        entity_by_type[ent_type] = []
    entity_by_type[ent_type].append(val.strip())

target_entities = ["SKILL", "UNIVERSITY", "COMPANY", "JOB_TITLE", "DEGREE", "DURATION"]
for ent_type in target_entities:
    if ent_type in entity_by_type:
        vals = entity_by_type[ent_type]
        # Chuẩn hóa về viết thường để đếm không trùng lắp do case-sensitive
        vals_lower = [v.lower() for v in vals]
        counts = Counter(vals_lower)
        
        print(f"\n🔥 Top 8 thực thể '{ent_type}' xuất hiện nhiều nhất:")
        for rank, (item, count) in enumerate(counts.most_common(8), 1):
            # Tìm giá trị viết hoa/thường nguyên bản phổ biến nhất
            original_representations = [v for v in vals if v.lower() == item]
            most_common_orig = Counter(original_representations).most_common(1)[0][0]
            print(f"  {rank}. {most_common_orig} ({count} lần)")

### 5.3. Trực quan hóa Kỹ năng (SKILL) xuất hiện nhiều nhất

In [ ]:
skills = entity_by_type.get("SKILL", [])
skills_lower = [s.lower() for s in skills]
skill_counts = Counter(skills_lower)

top_skills = []
for item, count in skill_counts.most_common(20):
    original_representations = [v for v in skills if v.lower() == item]
    most_common_orig = Counter(original_representations).most_common(1)[0][0]
    top_skills.append((most_common_orig, count))

top_skills_df = pd.DataFrame(top_skills, columns=["Kỹ năng", "Tần suất"])

plt.figure(figsize=(14, 8))
sns.barplot(data=top_skills_df, x="Tần suất", y="Kỹ năng", hue="Kỹ năng", palette="mako", legend=False)
plt.title("Top 20 Kỹ Năng (SKILL) Xuất Hiện Nhiều Nhất Trong Tập Huấn Luyện", fontsize=15, fontweight="bold", pad=15)
plt.xlabel("Số lần xuất hiện", fontsize=12)
plt.ylabel("Tên Kỹ Năng", fontsize=12)
plt.tight_layout()
plt.show()

### 5.4. Đám mây từ khóa Kỹ năng (Skill WordCloud)
Chúng ta sẽ tạo WordCloud cho các kỹ năng của ứng viên để xem bức tranh tổng quan kỹ thuật.

In [ ]:
try:
    from wordcloud import WordCloud
    
    # Gộp tất cả kỹ năng thành một chuỗi văn bản
    skill_text = " ".join(skills_lower)
    
    wordcloud = WordCloud(
        width=800, 
        height=400, 
        background_color="white",
        colormap="viridis",
        max_words=100,
        contour_width=3,
        contour_color='steelblue'
    ).generate_from_frequencies(skill_counts)
    
    plt.figure(figsize=(15, 7.5))
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.title("Đám Mây Từ Khóa Kỹ Năng (Skill WordCloud) Từ Tập Dữ Liệu", fontsize=16, fontweight="bold", pad=20)
    plt.tight_layout(pad=0)
    plt.show()
except ImportError:
    print("Thư viện wordcloud chưa được cài đặt. Để hiển thị WordCloud, vui lòng cài đặt bằng lệnh: pip install wordcloud")

## 6. Nhận xét và Đề xuất Hướng đi Tiếp theo

Dựa trên kết quả phân tích bộ dữ liệu huấn luyện `train.jsonl`, chúng ta rút ra một số kết luận quan trọng:

### 6.1. Nhận xét Về Chất lượng Dữ liệu
1. **Độ ổn định cấu trúc**: Hãy xem kết quả kiểm tra ở Mục 3.1. Nếu có lỗi lệch tokens và tags, cần xử lý dứt điểm trước khi huấn luyện model. 
2. **Nhất quán quy tắc BIO**: Kết quả ở Mục 3.2 cho thấy lượng vi phạm BIO (ví dụ: `I-` xuất hiện sau `O`). Điều này có thể khiến model gặp khó khăn khi học biên của các thực thể. Khuyến nghị chạy script dọn dẹp (BIO alignment) để chuẩn hóa lại.

### 6.2. Nhận xét Về Nội dung và Phân bố Nhãn
1. **Mất cân bằng nhãn (Label Imbalance)**:
   - Nhãn `SKILL` xuất hiện vượt trội hơn hẳn so với các nhãn khác.
   - Một số nhãn như `LANGUAGE` hoặc `MAJOR` có lượng dữ liệu gán nhãn rất ít. Điều này có thể khiến mô hình NER dự đoán kém ở các nhãn này (Recall thấp).
2. **Độ lệch ngôn ngữ (Language Bias)**:
   - Số lượng CV Tiếng Anh chiếm ưu thế tuyệt đối (>90%). Nếu hệ thống cần phân tích lượng lớn CV Tiếng Việt, mô hình sẽ không đạt hiệu quả tốt trên CV Tiếng Việt do thiếu dữ liệu học. Cần bổ sung thêm CV Tiếng Việt được gán nhãn.
3. **Sự không đồng đều về độ dài**:
   - Biểu đồ phân phối độ dài cho thấy độ dài CV dao động cực kỳ lớn (từ vài chục token đến hơn 1500 token). Đối với các model họ BERT như PhoBERT (giới hạn 256 hoặc 512 tokens), các CV quá dài sẽ bị cắt cụt (truncation), dẫn đến việc mất mát thông tin thực thể ở nửa sau của CV. Cần cân nhắc giải pháp chia nhỏ CV dài thành các đoạn nhỏ hơn (sentence/paragraph splitting) trong quá trình huấn luyện và suy luận.